# Point-pattern second-order statistics lab
Explore nearest-neighbor distances, homogeneous and inhomogeneous rectangular translation-corrected Ripley's K, and pair correlation. All coordinates are planar and use the same linear unit. The inhomogeneous estimator requires externally supplied intensity estimates at the observed points.


In [ ]:
import sys
sys.path.insert(0, "implementations/python/point-process")
from point_patterns import nearest_neighbor_distances, nearest_neighbor_cdf, ripley_k_translation, ripley_k_inhomogeneous_translation, pair_correlation_from_k, quadrat_count_test
import numpy as np
from spatial_scan import poisson_spatial_scan, monte_carlo_poisson_scan
import math

points = [(2,2), (3,2), (8,7), (8,8), (5,5), (6,5)]
radii = [0.5, 1, 2, 3, 4]
print('Nearest-neighbor distances:', nearest_neighbor_distances(points))
print('Nearest-neighbor CDF:', nearest_neighbor_cdf(points, radii))
k = ripley_k_translation(points, (0,0,10,10), radii)
print('K(r):', k)
print('Poisson reference:', [math.pi*r*r for r in radii])
print('Approximate g(r):', pair_correlation_from_k, quadrat_count_test(radii, k))


## Quadrat counts: a grid-scale CSR diagnostic

The conditional Monte Carlo test compares equal-area quadrat counts with a uniform multinomial null, given the observed total point count. The grid is a modelling choice: different grid sizes or alignments can produce different results.

In [ ]:
quadrat = quadrat_count_test(points, (0, 0, 10, 10), nx=3, ny=3, n_simulations=499, seed=2026)
print('counts by row:', quadrat['counts'])
print('expected per quadrat:', quadrat['expected_per_quadrat'])
print('Pearson statistic:', quadrat['pearson_statistic'])
print('conditional Monte Carlo p-value:', quadrat['pvalue'])


## Inhomogeneous K: intensity reweighting

Here the intensity values are supplied at the observed points. They are illustrative inputs, not estimates fitted from this tiny pattern. In real analysis, fit a defensible trend or use a leave-one-out estimator and assess sensitivity to its bandwidth.

In [ ]:
intensity_at_points = [0.012, 0.018, 0.010, 0.014, 0.020, 0.016]
k_inhom = ripley_k_inhomogeneous_translation(points, (0,0,10,10), radii, intensity_at_points)
print('Inhomogeneous K(r):', k_inhom)
print('Inhomogeneous Poisson reference:', [math.pi*r*r for r in radii])
print('Approximate inhomogeneous g(r):', pair_correlation_from_k(radii, k_inhom))


## Interpretation and cautions

1. Why must the intensity be evaluated at each observed point?
2. What happens if the fitted intensity surface absorbs genuine clustering?
3. Why does the translation correction depend on pair displacement?
4. Why should the observed K curve be compared with simulations under a fitted null model?

The implementation assumes a rectangular window and second-order intensity-reweighted stationarity. It does not estimate intensity, and a visually close match to the Poisson reference is not a formal test.

## Circular Poisson spatial scan

This baseline searches centroid-centred circles for elevated case rates while using population as the exposure offset. The example is synthetic and uses projected planar coordinates.

In [ ]:
scan_coordinates = [(0,0), (1,0), (10,0), (11,0), (20,0)]
scan_cases = [9, 8, 1, 1, 1]
scan_population = [100, 100, 100, 100, 100]
scan = poisson_spatial_scan(scan_coordinates, scan_cases, scan_population, max_population_fraction=0.5)
print('most likely zone:', scan['member_indices'])
print('log likelihood ratio:', scan['log_likelihood_ratio'])
print('relative risk:', scan['relative_risk'])
mc = monte_carlo_poisson_scan(scan_coordinates, scan_cases, scan_population, n_simulations=99, rng=np.random.default_rng(7))
print('conditional multinomial Monte Carlo p-value (99 simulations):', mc['p_value'])


The scan conditions on total cases and redistributes them across areas in proportion to population under the null. It searches circles centred on the supplied centroids and is limited by the maximum population fraction. It is not covariate-adjusted, time-dependent, elliptical or a full SaTScan implementation. The 99-simulation example is illustrative, not a high-resolution inferential analysis.

## Questions
1. How does the translation correction change pairs near the window boundary?
2. Why is K(r) cumulative while g(r) is local in scale?
3. What changes when the point intensity varies across the study window?
4. Why does a population offset matter for the spatial scan statistic?
5. Why is a Monte Carlo p-value needed after maximising over many candidate circles?
